# Hardware y Escala — Módulo 15

Este notebook es el compañero práctico del módulo de arquitectura. Aquí vas a:

1. Inspeccionar el hardware de tu propia máquina desde Python
2. Ver la diferencia de vectorización con tus propios ojos (y temporizador)
3. Observar efectos de caché en código real
4. Medir FLOPs empíricamente en multiplicaciones de matrices
5. Reproducir las gráficas del módulo y explorar los datos
6. Estimar costos de entrenamiento de LLMs interactivamente

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/15_arquitectura_de_computadoras/code/01_arquitectura.ipynb)

In [1]:
%pip install numpy matplotlib psutil -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import platform
import psutil
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')  # para compatibilidad en Colab
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

print(f"Python:    {platform.python_version()}")
print(f"NumPy:     {np.__version__}")
print(f"Plataforma: {platform.system()} {platform.machine()}")

Python:    3.13.1
NumPy:     1.26.4
Plataforma: Darwin arm64


---
## Sección 1: Tu hardware

Python puede interrogar al sistema operativo sobre el hardware disponible.
Esto es útil para entender en qué entorno corre tu código y qué puedes esperar de él.

In [3]:
# Información de CPU
print("=== CPU ===")
print(f"Arquitectura:       {platform.machine()}")
print(f"Procesador:         {platform.processor() or 'N/D (ver /proc/cpuinfo)'}")
print(f"Núcleos físicos:    {psutil.cpu_count(logical=False)}")
print(f"Núcleos lógicos:    {psutil.cpu_count(logical=True)}")
freq = psutil.cpu_freq()
if freq:
    print(f"Frecuencia actual:  {freq.current:.0f} MHz")
    print(f"Frecuencia máx:     {freq.max:.0f} MHz")

=== CPU ===
Arquitectura:       arm64
Procesador:         arm
Núcleos físicos:    8
Núcleos lógicos:    8
Frecuencia actual:  3228 MHz
Frecuencia máx:     3228 MHz


In [4]:
# Información de memoria y almacenamiento
print("=== Memoria ===")
ram = psutil.virtual_memory()
print(f"RAM total:          {ram.total / 2**30:.1f} GB")
print(f"RAM disponible:     {ram.available / 2**30:.1f} GB")
print(f"RAM en uso:         {ram.percent:.1f}%")

print("\n=== Almacenamiento ===")
for part in psutil.disk_partitions()[:2]:
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"{part.mountpoint}: {usage.total / 2**30:.0f} GB total, "
              f"{usage.free / 2**30:.0f} GB libres")
    except PermissionError:
        pass

=== Memoria ===
RAM total:          16.0 GB
RAM disponible:     2.9 GB
RAM en uso:         82.1%

=== Almacenamiento ===
/: 460 GB total, 16 GB libres
/System/Volumes/VM: 460 GB total, 16 GB libres


In [21]:
# ¿Hay GPU disponible?
print("=== GPU ===")
try:
    import torch
    if torch.cuda.is_available():
        print(f"CUDA disponible: SÍ")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        mem = torch.cuda.get_device_properties(0).total_memory
        print(f"VRAM: {mem / 2**30:.1f} GB")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        print("Metal (Apple GPU): disponible")
    else:
        print("Sin GPU acelerada disponible — usando CPU")
except ImportError:
    print("PyTorch no instalado — no se puede verificar GPU")
    print("En Colab: Runtime > Change runtime type > GPU para activarla")

=== GPU ===
PyTorch no instalado — no se puede verificar GPU
En Colab: Runtime > Change runtime type > GPU para activarla


> **💡 Prueba esto:** Compara tus resultados con los de alguien más en la clase.
> ¿Cuántos núcleos físicos vs lógicos tiene cada máquina? ¿Cuánta RAM?
> En Colab (CPU runtime): ¿cuántos núcleos obtienes? ¿Cuánto RAM?

---
## Sección 2: Vectorización — ver la diferencia

Esta es la demostración más importante del notebook.
Vamos a hacer exactamente la misma operación de tres formas y medir el tiempo.

In [6]:
N = 10_000_000
a = list(range(N))          # lista Python
b = list(range(N, 2 * N))

a_np = np.arange(N, dtype=np.float64)  # array NumPy
b_np = np.arange(N, N * 2, dtype=np.float64)

print(f"N = {N:,}")
print(f"Listas Python: {sum(len(str(x)) for x in [a, b])} objetos")
print(f"Arrays NumPy:  {a_np.nbytes / 1e6:.0f} MB + {b_np.nbytes / 1e6:.0f} MB")

N = 10,000,000
Listas Python: 188888890 objetos
Arrays NumPy:  80 MB + 80 MB


In [7]:
# Método 1: loop Python puro
t0 = time.perf_counter()
result_loop = [a[i] + b[i] for i in range(N)]
t_loop = time.perf_counter() - t0
print(f"Loop Python:    {t_loop:.3f} s")

Loop Python:    2.011 s


In [8]:
# Método 2: NumPy vectorizado
t0 = time.perf_counter()
result_np = a_np + b_np
t_numpy = time.perf_counter() - t0
print(f"NumPy:          {t_numpy:.4f} s")

# Verificar que el resultado es el mismo
assert np.allclose(result_loop[:100], result_np[:100]), "Los resultados difieren"

NumPy:          0.1442 s


In [9]:
# Comparación
speedup = t_loop / t_numpy
print(f"\n=== Resumen ===")
print(f"Loop Python:   {t_loop:.3f} s")
print(f"NumPy:         {t_numpy:.4f} s")
print(f"Speedup:       {speedup:.0f}×")
print(f"\nNumPy es {speedup:.0f} veces más rápido con los mismos datos y la misma operación.")
print("La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.")


=== Resumen ===
Loop Python:   2.011 s
NumPy:         0.1442 s
Speedup:       14×

NumPy es 14 veces más rápido con los mismos datos y la misma operación.
La diferencia viene de: SIMD, sin overhead de Python objects, memoria contigua.


> **💡 Prueba esto:**
> - Cambia `N` a 100_000 y a 100_000_000. ¿Cómo cambia el speedup?
> - Prueba con `np.float32` en vez de `np.float64`. ¿Es más rápido?
> - ¿Qué pasa si haces `a_np * b_np` (multiplicación) en vez de suma?

---
## Sección 3: Efectos de caché

Acceder a datos en orden secuencial (cache-friendly) es dramáticamente más rápido
que acceder de forma aleatoria, aunque el número de operaciones sea idéntico.

Esto no es sobre Python — es sobre física del hardware.

In [10]:
# Creamos una matriz cuadrada grande
SIZE = 2000
M = np.random.randn(SIZE, SIZE).astype(np.float32)
print(f"Matriz: {SIZE}×{SIZE} = {SIZE*SIZE:,} elementos")
print(f"Tamaño en memoria: {M.nbytes / 1e6:.1f} MB")

Matriz: 2000×2000 = 4,000,000 elementos
Tamaño en memoria: 16.0 MB


In [11]:
# Acceso secuencial: fila por fila (cache-friendly en NumPy/C)
# NumPy almacena matrices en row-major order: la fila i está contigua en memoria
t0 = time.perf_counter()
total = 0.0
for i in range(SIZE):
    total += M[i, :].sum()  # leer fila completa — datos contiguos en RAM
t_row = time.perf_counter() - t0
print(f"Acceso por filas:    {t_row:.4f} s  (total={total:.2f})")

Acceso por filas:    0.0075 s  (total=-596.48)


In [12]:
# Acceso por columnas (menos cache-friendly)
# Columna i: los elementos están separados por SIZE*4 bytes entre sí
t0 = time.perf_counter()
total2 = 0.0
for j in range(SIZE):
    total2 += M[:, j].sum()  # leer columna completa — saltos en memoria
t_col = time.perf_counter() - t0
print(f"Acceso por columnas: {t_col:.4f} s  (total={total2:.2f})")

print(f"\nRatio columnas/filas: {t_col/t_row:.2f}×")
print("(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)")

Acceso por columnas: 0.0111 s  (total=-596.48)

Ratio columnas/filas: 1.48×
(Un ratio > 1 indica efectos de caché, aunque la operación matemática es idéntica)


### ¿Por qué ocurre esto?

Cuando NumPy lee `M[i, :]` (una fila), los datos están contiguos en memoria —
el procesador los carga en caché en bloques de 64 bytes (cache lines).
Las siguientes lecturas ya están en caché.

Cuando lee `M[:, j]` (una columna), cada elemento está a `SIZE × 4 = 8,000 bytes`
del anterior. Cada acceso es un cache miss: hay que ir a buscar a RAM.

```
Fila [i, :]:   [0][1][2][3][4]...  ← contiguos, 1 cache miss por bloque
Columna [:, j]: [0]....[1]....[2].. ← separados, 1 cache miss por elemento
```

En pandas: `.iterrows()` accede por filas a un DataFrame que internamente
almacena columnas. Por eso es especialmente lento.

> **💡 Prueba esto:**
> - Cambia `SIZE` a 200. ¿Desaparece el efecto? (La matriz cabe en caché L3)
> - Prueba con `SIZE = 5000`. ¿Se amplifica el efecto?
> - ¿Qué pasa si conviertes la columna a array contiguo antes: `M[:, j].copy().sum()`?

---
## Sección 4: Multiplicación de matrices y FLOPs

La operación central de las redes neuronales. Vamos a medirla empíricamente
y calcular cuántos FLOPs realmente estamos ejecutando.

In [13]:
def benchmark_matmul(N, reps=3):
    """Mide tiempo de N×N matmul y calcula GFLOPS observados."""
    A = np.random.randn(N, N).astype(np.float32)
    B = np.random.randn(N, N).astype(np.float32)

    # Warm-up
    _ = A @ B

    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        C = A @ B
        times.append(time.perf_counter() - t0)

    t_med = np.median(times)
    flops = 2 * N**3           # FLOPs teóricos: 2×M×N×K con M=N=K
    gflops = flops / t_med / 1e9
    return t_med, gflops

sizes = [64, 128, 256, 512, 1024, 2048]
print(f"{'N':>6}  {'Tiempo (ms)':>12}  {'GFLOPS obs.':>12}")
print("-" * 36)
results = []
for N in sizes:
    t, g = benchmark_matmul(N)
    results.append((N, t, g))
    print(f"{N:>6}  {t*1000:>12.2f}  {g:>12.1f}")

     N   Tiempo (ms)   GFLOPS obs.
------------------------------------
    64          0.01          69.5
   128          0.01         625.3
   256          0.03        1091.2
   512          0.20        1362.0
  1024          1.29        1660.5
  2048         10.71        1603.6


In [14]:
# Graficar GFLOPS observados vs N
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), facecolor="#1a1a2e")

for ax in [ax1, ax2]:
    ax.set_facecolor("#16213e")
    for sp in ax.spines.values():
        sp.set_edgecolor("#2a2a4e")
    ax.tick_params(colors="#e0e0e0")
    ax.xaxis.label.set_color("#e0e0e0")
    ax.yaxis.label.set_color("#e0e0e0")
    ax.title.set_color("#e0e0e0")
    ax.yaxis.grid(True, color="#2a2a4e", alpha=0.5)
    ax.set_axisbelow(True)

ns     = [r[0] for r in results]
times  = [r[1] * 1000 for r in results]  # ms
gflops = [r[2] for r in results]

ax1.plot(ns, times, 'o-', color="#0db7ed", linewidth=2, markersize=7)
ax1.set_xlabel("Tamaño de matriz N")
ax1.set_ylabel("Tiempo (ms)")
ax1.set_title("Tiempo de N×N matmul")

ax2.plot(ns, gflops, 's-', color="#f0a500", linewidth=2, markersize=7)
ax2.set_xlabel("Tamaño de matriz N")
ax2.set_ylabel("GFLOPS observados")
ax2.set_title("GFLOPS observados — crece con N")

fig.suptitle("Multiplicación de matrices: tiempo y GFLOPS",
             color="#e0e0e0", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("matmul_benchmark.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()
print("\n¿Por qué los GFLOPS crecen con N?")
print("Con matrices más grandes, la GPU (o BLAS) puede saturar sus núcleos.")
print("Matrices pequeñas tienen overhead de setup relativo mayor.")


¿Por qué los GFLOPS crecen con N?
Con matrices más grandes, la GPU (o BLAS) puede saturar sus núcleos.
Matrices pequeñas tienen overhead de setup relativo mayor.


/var/folders/kc/y3bhhm_n1579pfjvxxnk_mvc0000gn/T/ipykernel_1211/1733488152.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **💡 Prueba esto:**
> - Compara `np.float32` vs `np.float64`. ¿Cuánto cambia el rendimiento?
> - Si tienes GPU disponible: prueba con `torch.matmul` en CUDA y compara.
> - ¿Qué GFLOPS teóricos tiene tu CPU? (Busca el modelo y sus specs.)

---
## Sección 5: Reproduce el gráfico histórico de FLOPs

Los datos del módulo, reproducibles y modificables.

In [15]:
# Datos históricos: (año, nombre, USD por GFLOP)
# Fuentes: diversos estudios de Karl Rupp, Our World in Data, benchmarks públicos
flops_history = [
    (1961, "IBM 7090",           1e10),
    (1984, "Cray X-MP",          4.2e7),
    (1994, "Intel Pentium",      3e4),
    (1997, "Intel Pentium II",   1e3),
    (2001, "AMD Athlon XP",      1e2),
    (2006, "PlayStation 3",      1.0),
    (2008, "AMD Radeon HD 4870", 0.065),
    (2013, "NVIDIA GTX 780",     0.002),
    (2017, "NVIDIA GTX 1080",    3.5e-4),
    (2020, "NVIDIA RTX 3090",    4e-5),
    (2022, "NVIDIA RTX 4090",    1.5e-5),
    (2023, "NVIDIA H100",        2e-6),
]

years = [d[0] for d in flops_history]
costs = [d[2] for d in flops_history]
names = [d[1] for d in flops_history]

reduction = flops_history[0][2] / flops_history[-1][2]
print(f"Reducción total (1961 → 2023): {reduction:,.0f}×")

Reducción total (1961 → 2023): 5,000,000,000,000,000×


In [17]:
fig, ax = plt.subplots(figsize=(12, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.set_axisbelow(True)

# Colorear: CPU = azul, GPU = violeta, consola = verde
colors = [
    "#0db7ed", "#0db7ed", "#0db7ed", "#0db7ed", "#0db7ed",  # CPUs
    "#2ecc71",                                                # PS3
    "#892ca0", "#892ca0", "#892ca0", "#892ca0", "#892ca0", "#892ca0",  # GPUs
]

ax.plot(years, costs, color="#444466", linewidth=1, alpha=0.5, zorder=1)
for i, (yr, cost, name) in enumerate(zip(years, costs, names)):
    ax.scatter(yr, cost, color=colors[i], s=80, zorder=3,
               edgecolors="white", linewidths=0.5)
    ax.annotate(name, (yr, cost),
                xytext=(5, 3), textcoords="offset points",
                color="#e0e0e0", fontsize=8)

ax.set_yscale("log")
ax.set_xlabel("Año")
ax.set_ylabel("USD por GFLOP (escala log)")
ax.set_title(f"Costo histórico de 1 GFLOP — {reduction:,.0f}× más barato en 62 años",
             fontweight="bold")

import matplotlib.patches as mpatches
leg = [
    mpatches.Patch(color="#0db7ed", label="CPU"),
    mpatches.Patch(color="#892ca0", label="GPU"),
    mpatches.Patch(color="#2ecc71", label="Consola"),
]
ax.legend(handles=leg, facecolor="#16213e", labelcolor="#e0e0e0", edgecolor="#2a2a4e")

plt.tight_layout()
plt.savefig("flops_historico_notebook.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

/var/folders/kc/y3bhhm_n1579pfjvxxnk_mvc0000gn/T/ipykernel_1211/3708879331.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **💡 Prueba esto:**
> - Agrega chips nuevos que encuentres online (MI300X, RTX 5090, etc.)
> - Calcula el precio del chip actual y divídelo entre los TFLOPS para obtener USD/GFLOP
> - ¿Qué costo por GFLOP tiene tu propia computadora?

---
## Sección 6: Explorador de costos de LLMs

Una función para estimar cuánto costaría entrenar un modelo dado.

In [18]:
def estimar_entrenamiento(
    params_B: float,          # parámetros del modelo en billones
    tokens_B: float,          # tokens de entrenamiento en billones
    gpu_tflops_bf16: float,   # rendimiento efectivo del GPU en TFLOPS BF16
    gpu_price_per_hour: float = 3.0,  # USD por hora por GPU
    gpu_efficiency: float = 0.35,     # eficiencia real (35% típico en clúster)
) -> dict:
    """
    Estima FLOPs, tiempo y costo de entrenamiento usando la fórmula de Chinchilla:
    FLOPs ≈ 6 × parámetros × tokens
    """
    params = params_B * 1e9
    tokens = tokens_B * 1e9

    flops_total = 6 * params * tokens

    # FLOPS/s efectivos de 1 GPU
    flops_per_sec_gpu = gpu_tflops_bf16 * 1e12 * gpu_efficiency

    tiempo_segundos = flops_total / flops_per_sec_gpu
    tiempo_horas = tiempo_segundos / 3600
    costo_usd = tiempo_horas * gpu_price_per_hour

    return {
        "flops_total": flops_total,
        "flops_total_str": f"{flops_total:.2e}",
        "gpu_horas": tiempo_horas,
        "costo_1_gpu_usd": costo_usd,
        "gpus_para_1_semana": tiempo_horas / (24 * 7),
    }

print("Función lista. Probemos con algunos ejemplos...")

Función lista. Probemos con algunos ejemplos...


In [19]:
# H100 con ~1,000 TFLOPS BF16 (rendimiento pico)
H100_TFLOPS = 1000
H100_PRICE  = 3.0   # USD/hora (cloud spot)

ejemplos = [
    ("LLaMA 2 7B",   7,    1_400),   # 7B params, ~1.4T tokens
    ("LLaMA 2 70B",  70,   2_000),   # 70B params, 2T tokens
    ("GPT-3 175B",   175,  300),     # 175B params, 300B tokens
    ("Modelo 1B (tuyo)", 1, 20),     # 1B params, 20B tokens — un proyecto pequeño
]

print(f"{'Modelo':<22} {'FLOPs':>10}  {'GPU-horas':>12}  {'Costo 1 GPU':>14}  {'GPUs p/semana':>14}")
print("-" * 78)

for nombre, params_B, tokens_B in ejemplos:
    r = estimar_entrenamiento(params_B, tokens_B, H100_TFLOPS, H100_PRICE)
    print(f"{nombre:<22} {r['flops_total_str']:>10}  "
          f"{r['gpu_horas']:>12,.0f}  "
          f"${r['costo_1_gpu_usd']:>12,.0f}  "
          f"{r['gpus_para_1_semana']:>14,.0f}")

Modelo                      FLOPs     GPU-horas     Costo 1 GPU   GPUs p/semana
------------------------------------------------------------------------------
LLaMA 2 7B               5.88e+22        46,667  $     140,000             278
LLaMA 2 70B              8.40e+23       666,667  $   2,000,000           3,968
GPT-3 175B               3.15e+23       250,000  $     750,000           1,488
Modelo 1B (tuyo)         1.20e+20            95  $         286               1


In [20]:
# Visualizar: parámetros vs costo estimado
modelos_viz = [
    {"name": "BERT-Large",   "params_B": 0.34,  "flops": 1.5e20, "cost_M": 0.007},
    {"name": "GPT-2",        "params_B": 1.5,   "flops": 5e19,   "cost_M": 0.05},
    {"name": "T5-11B",       "params_B": 11,    "flops": 5e21,   "cost_M": 0.5},
    {"name": "GPT-3 175B",   "params_B": 175,   "flops": 3.1e23, "cost_M": 5},
    {"name": "PaLM 540B",    "params_B": 540,   "flops": 2.5e24, "cost_M": 50},
    {"name": "LLaMA 2 70B",  "params_B": 70,    "flops": 2e24,   "cost_M": 20},
    {"name": "GPT-4 (est.)", "params_B": 1800,  "flops": 2e25,   "cost_M": 100},
]

fig, ax = plt.subplots(figsize=(10, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
for sp in ax.spines.values():
    sp.set_edgecolor("#2a2a4e")
ax.tick_params(colors="#e0e0e0")
ax.xaxis.label.set_color("#e0e0e0")
ax.yaxis.label.set_color("#e0e0e0")
ax.title.set_color("#e0e0e0")
ax.yaxis.grid(True, color="#2a2a4e", alpha=0.4)
ax.xaxis.grid(True, color="#2a2a4e", alpha=0.2)
ax.set_axisbelow(True)

colors_m = plt.cm.plasma(np.linspace(0.1, 0.9, len(modelos_viz)))
for i, m in enumerate(modelos_viz):
    ax.scatter(m["params_B"], m["cost_M"], s=100, color=colors_m[i],
               zorder=3, edgecolors="white", linewidths=0.5)
    ax.annotate(m["name"], (m["params_B"], m["cost_M"]),
                xytext=(6, 4), textcoords="offset points",
                color="#e0e0e0", fontsize=9)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Parámetros (billones)")
ax.set_ylabel("Costo de entrenamiento (millones USD)")
ax.set_title("Parámetros vs Costo de entrenamiento", fontweight="bold")

plt.tight_layout()
plt.savefig("llm_costo_notebook.png", dpi=120, bbox_inches="tight", facecolor="#1a1a2e")
plt.show()

/var/folders/kc/y3bhhm_n1579pfjvxxnk_mvc0000gn/T/ipykernel_1211/2484734864.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **💡 Prueba esto:**
> - Llama a `estimar_entrenamiento` con tu propio modelo hipotético.
> - ¿Cuántas GPUs H100 necesitarías para terminar en 1 semana un modelo de 13B con 1T tokens?
> - ¿Qué pasa si cambias la eficiencia de 35% a 50%? ¿Qué hace que no sea 100%?

---
## Ejercicio integrador

Tienes acceso a un clúster con 8 GPUs H100 (1,000 TFLOPS BF16 cada una, rendimiento
efectivo 35%, $3/hora cada una). Quieres entrenar un modelo de lenguaje.

In [ ]:
# Tu código aquí

# 1. Calcula cuántos parámetros puedes entrenar en máximo 1 semana con 8 GPUs
#    si usas el ratio óptimo de Chinchilla (20 tokens por parámetro).

# 2. ¿Cuánto cuesta ese entrenamiento en total?

# 3. Si el presupuesto máximo es $10,000 USD, ¿qué tamaño de modelo puedes entrenar?
#    (sigue asumiendo Chinchilla y 8 GPUs H100)

# 4. Benchmarkea en tu máquina: ¿cuántos GFLOPS obtienes en matmul 1024×1024?
#    ¿Cuánto tiempo tardaría en tu CPU hacer lo que un H100 hace en 1 segundo?

# Pista para el punto 1:
# flops_totales = 6 × params × tokens
# tokens = 20 × params  (Chinchilla)
# tiempo_disponible = 8 GPUs × 7 días × 24 horas × 3600 segundos × TFLOPS × eficiencia
# despeja params de: flops_totales ≤ tiempo_disponible × FLOPS/s

<details>
<summary>Pista para el punto 1</summary>

```python
n_gpus = 8
h100_tflops = 1000
eficiencia = 0.35
semanas = 1

flops_disponibles = n_gpus * h100_tflops * 1e12 * eficiencia * semanas * 7 * 24 * 3600

# FLOPs = 6 × params × 20 × params = 120 × params²
# params = sqrt(FLOPs / 120)
import math
params = math.sqrt(flops_disponibles / 120)
print(f"Parámetros máximos: {params/1e9:.1f}B")
```
</details>

---
## Resumen

| Experimento | Lección |
|-------------|----------|
| Hardware inspection | Cada máquina tiene restricciones concretas: cores, RAM, ausencia de GPU |
| Vectorización | NumPy no es solo comodidad — usa instrucciones SIMD del hardware |
| Efectos de caché | El orden de acceso a memoria importa tanto como el número de operaciones |
| MatMul FLOPs | Matrices grandes saturan el hardware; pequeñas son overhead-bound |
| FLOPs histórico | El costo de cómputo cayó 5 billones de veces en 62 años |
| Escala LLMs | GPT-4 costó ~$100M en compute. LLaMA 2 7B: estimado en ~$3M. Fine-tuning: miles. |

**El mensaje central:** el hardware no es un detalle de implementación.
Es el presupuesto físico dentro del cual vive cada algoritmo.